# Who This Book is For

Whether you're an academic, a professional, or a complete beginner, this book offers practical causal inference tools for anyone who uses data to answer questions.

This book has been written to integrate causal tools into the standard data science workflow.
No prior knowledge of causal methods is required.
Though a few prerequisites will help you get the most out of it:

* Understanding of standard statistical and machine learning methods: correlation, linear regression, generalised linear models, tree-based models
* Familiarity with Python for case studies, illustrations, and implementations; experience using the standard PyData stack (Pandas, NumPy) will be useful.

The book builds on these foundations to provide a practical guide to causal inference in real-world settings.


## Correlation

To understand causality, you must first master **correlation**, which measures the statistical association between two variables.
In applied inference, we typically use Pearson's correlation coefficient ($r$) to quantify the strength and direction of a linear relationship, ranging from *-1* to *+1*.
While 0 means mo linear relationship, +1 (-1) mean perfect positive (negative) linear relationship and results can be interpreted as weak if $r$ is in 0.1–0.3, moderate if in 0.3–0.5, and  strong if >0.5.
It measures the linear relationship strength between two variables, and it is calculated by dividing the covariance of variables $X$ and $Y$ by the product of their standard deviation ($\sigma$):

$$r = \frac{cov(X, Y)}{\sigma_X \sigma_Y}$$

where covariance can be expressed in terms of expectation

$$cov(X, Y) = \mathbb{E}[ (X - \mu_X)(Y - \mu_Y) ] $$

However, for causal analysis, the most critical concept to "fix" is that correlation is *nondirectional*.
If $X$ and $Y$ are correlated, the data alone cannot tell you if $X$ causes $Y$, $Y$ causes $X$, or if a third "confounding" variable $Z$ causes both.
In causal modelling, we treat correlation as the "raw material" that must be decomposed into **causal effects** and **spurious associations** (bias) using tools like Directed Acyclic Graphs (DAGs).

##### Key Concepts

- **Linearity**
: Standard correlation only captures linear patterns. If your causal mechanism is non-linear, $r$ might be near zero even if a strong functional relationship exists.

- **Confounding**
: The presence of a correlation often masks a "back-door path." Causal inference aims to "block" these paths to see if a correlation persists.

- **Independence**
: If two variables are truly independent, their correlation is zero. Causal inference often relies on finding "conditional independence", where $X$ and $Y$ stop being correlated once you account for a specific set of other variables.


## Linear Regression

In causal inference, **linear regression** is the bridge between raw correlation and structural modelling.
While a simple correlation tells you that $X$ and $Y$ move together, a regression model attempts to quantify that relationship by fitting a line that minimises the sum of squared residuals.

$$y = \alpha + \beta x$$

The intercept ($\alpha$) is the predicted value of $y$ when $X=0$ while the slope ($\beta$) represents the expected change in $Y$ for a one-unit change in $X$.

$$\alpha = \overline{Y} - \beta \overline{X}$$

$$\beta = cov(X, Y) \frac{\sigma_Y}{\sigma_X}$$

However, in an applied causal context, a regression coefficient is not automatically a "causal effect."
It only achieves that status if you have successfully accounted for all confounders.
By using multiple linear regression, you "control" for other variables, effectively looking at the relationship between $X$ and $Y$ while holding those other factors constant.
This is the mathematical equivalent of "blocking" back-door paths in a causal graph.

##### Key Concepts

- **The Error Term ($\epsilon$)**
: In causal work, the error term isn't just "noise", it represents all unobserved factors affecting $Y$. For a coefficient to be causal, $X$ must be independent of $\epsilon$ (Exogeneity).

- **Ceteris Paribus**
: This is the "all else being equal" assumption. Regression allows us to isolate the impact of one variable while mathematically "fixing" others at their means.

- **Omitted Variable Bias (OVB)**
: If you fail to include a variable that affects both $X$ and $Y$, your regression coefficient will be "biased"; it will swallow up the effect of the missing variable, leading to a false causal conclusion.

> **A Note on "Bad" Models:** Don't fall into the trap of thinking a high $R^2$ (good fit) means you've found a causal link.
> You can have a perfect fit in a model that is causally meaningless if you've ignored the underlying structure.

## Generalised Linear Models (GLMs)

Standard linear regression assumes your outcome $Y$ follows a normal distribution and changes linearly with your predictors.
**Generalised Linear Models (GLMs)** break these constraints, allowing you to model causal effects on binary outcomes (e.g., "did the subject recover?"), counts, or proportions.
By using a **link function** (such as the logit or log) a GLM transforms the expected value of the outcome.
As a consequence, the linear combination of your predictors remains mathematically valid, even when the raw data is skewed or bounded.

In applied causal inference, GLMs are indispensable for estimating **Propensity Scores** (usually via Logistic Regression) and handling non-continuous treatment effects.
The "fix" here is understanding that while the underlying math changes, the causal logic remains the same: you are still trying to isolate the effect of $X$ on $Y$ while blocking back-door paths.
However, because GLMs are often non-linear, a one-unit change in $X$ doesn't have a constant effect on $Y$ across the entire scale, making the interpretation of coefficients slightly more complex than in simple Ordinary Least Squares (OLS).

##### Key Concepts

- **The Link Function**
: This is the "glue" that connects the linear predictor ($\beta X$) to the mean of the distribution. For causal questions with "yes/no" outcomes, the logit link is the standard for calculating odds ratios.

- **Distributional Assumptions**
: Unlike standard regression, GLMs require you to specify the distribution (the "family") of the outcome, such as Binomial, Poisson, or Gamma. Ensure your distribution choice aligns with your causal hypothesis.

- **Interpretability**
: Causal effects in GLMs are often expressed as odds ratios or incidence rate ratios rather than simple additive changes. Be clear about what your coefficient represents: it's not always a direct change in the outcome value.

> **A Note on Model Selection:** It is tempting to default to a logit model for everything categorical.
> However, if your goal is an Average Treatment Effect (ATE), sometimes a linear probability model is more robust and easier to interpret, provided you acknowledge its limitations.
> Don't overcomplicate your model if a simpler one provides a more reliable causal estimate.


## Tree-Based Models

While linear models rely on pre-defined equations, **tree-based models** (like Decision Trees, Random Forests, and Gradient Boosting) partition data into nested subsets based on feature thresholds.
In a causal context, their primary strength is capturing **non-linearity** and **complex interactions** without you having to specify them manually.
They excel at identifying **Heterogeneous Treatment Effects (HTE)**, essentially answering the question: _"Does this treatment work differently for different groups of people?"_

However, traditional trees are designed for prediction, not inference.
To "fix" them for causal work, you must move toward **Causal Trees** or **Honest Forests**.
The standard approach splits data to minimise prediction error, but a causal tree splits data to maximise the difference in the treatment effect between "leaves."
To avoid overfitting and biased estimates, "honesty" is required: one part of your data is used to determine the tree's structure (where the splits happen), and a completely separate part is used to estimate the actual effects within those splits.

##### Key Concepts

**Non-Parametric Flexibility**
: Unlike GLMs, trees don't assume a specific shape for the relationship. This is vital when the "back-door" variables you need to control for have messy, non-linear distributions.

**Interaction Discovery**
: Trees naturally find interactions (e.g., a drug works for men over 50 but not women under 30). This makes them the gold standard for "Personalised Medicine" or targeted policy interventions.

**The "Honesty" Requirement**
: You cannot use the same data to find a pattern and estimate its magnitude. If you do, your treatment effects will be wildly optimistic and likely false.

**Variable Importance**
: Trees can rank which variables are most "predictive" of the outcome, helping you identify potential confounders you might have missed in a simpler linear model.


# Contents by Chapter

**Part 1** introduces why causality is a useful tool and establishes core concepts, focusing on the "potential outcomes" framework.
It covers the complete estimation process from initial data questions to evaluating results; and concludes with an overview of causal discovery for learning structures from observational data:

* **Chapter 1** explains the importance of causality using examples like Simpson's paradox and spurious correlation.
  It highlights the shift from traditional machine learning to causal methods, provides a brief history of the field, and lists further learning resources.

* **Chapter 2** introduces the potential outcomes framework to explain the theoretical foundations of causal effects.
  Using a case study of an educational television program, it provides a practical look at cause-and-effect relationships.
  The chapter concludes with exercises from Gelman and Hill to help you model and apply these theories.
  [Link to case study in Google Colab](https://colab.research.google.com/drive/1zL_bugMw8BHI6Kc3fBoUh-tiQ18kz_17?usp=sharing).

* **Chapter 3** builds on the potential outcomes framework by introducing causal graphs as a practical way to infer relationships.
  It covers the general causal inference process, explores various estimation techniques, and provides a comparative analysis using the Lalonde dataset.
  [Link to case study in Google Colab](https://colab.research.google.com/drive/1e4Ab5pAyQlKBlS3URBDe9rYIS8Z9yjd-?usp=sharing).

* **Chapter 4** explores the challenges of building causal models in practice and introduces causal discovery as a way to learn structures from observational data.
  It outlines various algorithms, weighing their strengths and limitations, and concludes with a real-world case study to demonstrate how these techniques uncover causal relationships.
  [Link to case study in Google Colab](https://colab.research.google.com/drive/1Vb-GISvjprqYDwSoHxcgfq_Y1hP0c4bM?usp=sharing).

---

**Part 2** explores how causal inference integrates with other machine learning sub-domains, such as computer vision, natural language processing (NLP), and time-series analysis:

* **Chapter 5** explores causal inference in Natural Language Processing (NLP).
  It covers how to calculate causal effects when the treatment, outcome, or confounders consist of text data.
  The chapter includes a case study on predicting film revenue.
  [Link to case study in Google Colab](https://colab.research.google.com/drive/1E_4D2pjtEn0A2lD9tXEeux6Ewar0SzMA).

* **Chapter 6** examines the intersection of causality and computer vision.
  It addresses how causal methods can tackle spurious correlations and confounding in image data, specifically within image classification and visual question-answering.
  The chapter concludes with a case study on using causal techniques to improve model robustness via an adversarial transfer dataset.
  [Link to case study in Google Colab](https://colab.research.google.com/drive/1nOO77UGrcE9aUlaXwkEQB2QoPv8Y9v0l?usp=sharing).

* **Chapter 7** explores a recent method for time-dependent causal inference that is able to not only determine causation, but the temporal delay between the cause and effect variables.
  At present, this chapter does not describe time-dependent causal inference in the presence of confounding associations.
  We include a case study using an open-source bike sharing dataset.
  [Link to case study in Google Colab](https://colab.research.google.com/drive/1Lsa5DYGvxuiHfuVILNYggMWv3HHsMhCZ).

---

**Part 3** explores advanced topics within the field of causality:

* **Chapter 8** explores how causal inference can be used to assess and improve model fairness.
  It introduces algorithmic bias, compares non-causal measurements with causal alternatives, and addresses the specific confounders found in fairness settings.
  The chapter concludes with a case study on the COMPAS dataset, comparing these different approaches.
  [Link to case study in Google Colab](https://colab.research.google.com/drive/1bviQfw1BWtgKw4O-XXPAqXb-t7vtMRls?usp=sharing).

* **Chapter 9** provides an overview of how causality enhances reinforcement learning (RL).
  It covers techniques for improving world models, merging online and offline data, increasing sample efficiency, and explaining agent incentives.
  The chapter concludes by discussing the barriers to large-scale adoption of causal RL.
